<a href="https://colab.research.google.com/github/Amrutasai/azure-hello-world/blob/main/Telegram_Matrimony_filter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
uploaded = files.upload()


Saving messages.html to messages (1).html
Saving messages_1.html to messages_1.html
Saving messages_2.html to messages_2.html
Saving messages_3.html to messages_3.html
Saving messages_4.html to messages_4.html
Saving messages2.html to messages2 (1).html
Saving messages3.html to messages3 (1).html
Saving messages4.html to messages4 (1).html
Saving messages5.html to messages5 (1).html
Saving messages6.html to messages6 (1).html
Saving messages7.html to messages7 (1).html
Saving messages8.html to messages8 (1).html
Saving messages9.html to messages9 (1).html


In [3]:
!pip install beautifulsoup4 openpyxl
!pip install beautifulsoup4 pandas openpyxl

BSPD

In [5]:
import re
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
import glob

# ---------- CONFIG ----------
MAX_AGE = 28
MIN_SALARY = 15
TARGET_CITIES = ["mumbai", "thane", "navi mumbai"]

# 👉 NEW: TARGET COLLEGES
TARGET_COLLEGES = ["iit", "iim", "nit", 'bits']

CURRENT_YEAR = datetime.now().year


# ---------- COLLEGE ALIASES ----------
ALIASES = {
    "iit": ["iit", "indian institute of technology"],
    "iim": ["iim", "indian institute of management"],
    "nit": ["nit", "national institute of technology"],
    "bits": ["bits", "birla institute of technology"]
}


# ---------- HELPERS ----------

def extract_year(text):
    match = re.search(r'(19\d{2}|20\d{2})', text or "")
    return int(match.group(1)) if match else None


def calculate_age(year):
    return CURRENT_YEAR - year if year else None


def extract_dob(text):
    patterns = [
        r'DOB[:\s\-]*([^\n]+)',
        r'DoB[:\s\-]*([^\n]+)',
        r'Date of birth[:\s\-]*([^\n]+)',
    ]
    for p in patterns:
        m = re.search(p, text, re.IGNORECASE)
        if m:
            return m.group(1).strip()
    return None


def extract_salary(text):
    text = text.lower()
    match = re.search(r'(\d+(\.\d+)?)\s*(lpa|lac|lakh)', text)
    return float(match.group(1)) if match else None


def extract_name(text):
    match = re.search(r'Name[:\s\-]*([A-Za-z\s\.]+)', text, re.IGNORECASE)
    return match.group(1).strip() if match else None


def extract_contact(text):
    phones = re.findall(r'\b\d{10}\b', text)
    emails = re.findall(r'[\w\.-]+@[\w\.-]+', text)
    return ", ".join(phones), ", ".join(emails)


def extract_location(text):
    text_lower = text.lower()
    for city in TARGET_CITIES:
        if city in text_lower:
            return city.title()
    return None


def format_message(text_div):
    for br in text_div.find_all("br"):
        br.replace_with("\n")

    text = text_div.get_text()

    lines = [line.strip() for line in text.split("\n")]
    lines = [line for line in lines if line]

    return "\n".join(lines)


# ---------- NEW: COLLEGE MATCH ----------

def get_matching_college(text, target_colleges):
    text_lower = text.lower()

    for college in target_colleges:
        variants = ALIASES.get(college, [college])

        for v in variants:
            if re.search(rf'\b{v}\b', text_lower):
                return college.upper()

    return None


# ---------- FINAL FILTER LOGIC ----------

def is_female_profile(name, text):
    text_lower = text.lower()

    if name:
        name_lower = name.lower().strip()
        if name_lower.startswith(("ms ", "smt ", "ms. ", "smt. ")):
            return True

    female_emojis = ["👩", "👰", "👧", "🙎‍♀️", "👩🏻", "👩🏻‍💼", "👩🏻‍⚖️"]
    if any(e in text for e in female_emojis):
        return True

    if re.search(r'bride\s*details[:\-\.]?', text_lower):
        return True

    if re.search(
        r'the\s+information\s+given\s+by\s+me\s+about\s+my\s+daughter\s+is\s+true',
        text_lower,
        re.IGNORECASE
    ):
        return True

    if re.search(
        r'\bname\s*of\s*(the\s*)?bride\b\s*[:\-]?',
        text_lower,
        re.IGNORECASE
    ):
        return True

    return False


# ---------- PROCESS SINGLE FILE ----------

def process_html(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    results = []

    for msg in soup.find_all("div", class_="message"):
        text_div = msg.find("div", class_="text")

        if not text_div:
            continue

        text = format_message(text_div)

        # Only groom posts
        if "groom" not in text.lower():
            continue

        name = extract_name(text)

        # ❌ Remove female profiles
        if is_female_profile(name, text):
            continue

        # ---------- NEW: COLLEGE FILTER ----------
        matched_college = get_matching_college(text, TARGET_COLLEGES)

        if matched_college is None:
            continue

        dob_text = extract_dob(text)
        year = extract_year(dob_text)
        age = calculate_age(year)

        salary = extract_salary(text)
        location = extract_location(text)
        phones, emails = extract_contact(text)

        # ---------- STRICT FILTER ----------
        if age is None or age > MAX_AGE:
            continue

        if salary is not None and salary < MIN_SALARY:
            continue

        if location is None:
            continue

        results.append({
            "Name": name,
            "DOB": dob_text,
            "Age": age,
            "Salary (LPA)": salary,
            "Location": location,
            "College Match": matched_college,  # 👈 NEW
            "Phones": phones,
            "Emails": emails,
            "Message": text,
            "Source File": file_path
        })

    return results


# ---------- PROCESS ALL FILES ----------

files_list = glob.glob("message*.html")

print("Files detected:", files_list)

all_results = []

for file in files_list:
    print(f"Processing: {file}")
    file_results = process_html(file)
    all_results.extend(file_results)


# ---------- DEDUPLICATION ----------

seen = set()
unique_results = []

for row in all_results:
    key = (row["Name"], row["Phones"]) if row["Phones"] else (row["Name"],)

    if key not in seen:
        seen.add(key)
        unique_results.append(row)

data = unique_results


# ---------- OUTPUT ----------

print(f"\n✅ Total Unique Matches: {len(data)}\n")

for i, row in enumerate(data, 1):
    print(f"--- Match {i} ---")
    print(f"Name: {row['Name']}")
    print(f"Age: {row['Age']}")
    print(f"Salary: {row['Salary (LPA)']}")
    print(f"Location: {row['Location']}")
    print(f"College: {row['College Match']}")
    print(f"Source: {row['Source File']}")
    print("\n--- Message ---")
    print(row["Message"])
    print("\n" + "="*80 + "\n")


# ---------- EXPORT ----------
df = pd.DataFrame(data)
df.to_excel("final_matches.xlsx", index=False)

print("✅ Excel created: final_matches.xlsx")

Files detected: ['messages_3.html', 'messages2.html', 'messages9 (1).html', 'messages.html', 'messages6.html', 'messages8.html', 'messages9.html', 'messages3.html', 'messages_2.html', 'messages4.html', 'messages3 (1).html', 'messages_1.html', 'messages5 (1).html', 'messages2 (1).html', 'messages7.html', 'messages7 (1).html', 'messages5.html', 'messages_4.html', 'messages (1).html', 'messages6 (1).html', 'messages4 (1).html', 'messages8 (1).html']
Processing: messages_3.html
Processing: messages2.html
Processing: messages9 (1).html
Processing: messages.html
Processing: messages6.html
Processing: messages8.html
Processing: messages9.html
Processing: messages3.html
Processing: messages_2.html
Processing: messages4.html
Processing: messages3 (1).html
Processing: messages_1.html
Processing: messages5 (1).html
Processing: messages2 (1).html
Processing: messages7.html
Processing: messages7 (1).html
Processing: messages5.html
Processing: messages_4.html
Processing: messages (1).html
Processing

In [ ]:
from google.colab import files
files.download("final_matches.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
import glob
import os

for f in glob.glob("*.html"):
    os.remove(f)

print("✅ All HTML files removed")

for f in glob.glob("*.xlsx"):
    os.remove(f)

print("✅ All Excel files removed")

✅ All HTML files removed
✅ All Excel files removed


Yellapragada

In [7]:
from google.colab import files
uploaded = files.upload()


Saving messages.html to messages.html
Saving messages2.html to messages2.html
Saving messages3.html to messages3.html


In [10]:
import re
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime
import glob

# ---------- CONFIG ----------
MAX_AGE = 27
MIN_SALARY = 15
TARGET_CITIES = ["mumbai", "thane", "navi mumbai"]

TARGET_COLLEGES = ["iit", "iim", "nit", "bits", "iiit"]

CURRENT_YEAR = datetime.now().year


# ---------- HELPERS ----------
def format_message(text_div):
    for br in text_div.find_all("br"):
        br.replace_with("\n")

    text = text_div.get_text()
    lines = [line.strip() for line in text.split("\n")]
    lines = [line for line in lines if line]

    return "\n".join(lines)


def extract_field(text, field):
    match = re.search(rf'{field}[:.\s]*([^\n]+)', text, re.IGNORECASE)
    return match.group(1).strip() if match else None


def extract_year(text):
    match = re.search(r'(19\d{2}|20\d{2})', text or "")
    return int(match.group(1)) if match else None


def calculate_age(year):
    return CURRENT_YEAR - year if year else None


# ---------- SALARY PARSER ----------
def extract_salary(text):
    text = text.lower().replace(",", "")

    # LPA
    match = re.search(r'(\d+(\.\d+)?)\s*(lpa|lakhs|lakh|lac)', text)
    if match:
        return float(match.group(1))

    # Monthly → convert to LPA
    match = re.search(r'(\d+(\.\d+)?)\s*(per month|/month)', text)
    if match:
        return float(match.group(1)) * 12

    # Yearly INR → convert to LPA
    match = re.search(r'(\d{6,7})', text)
    if match:
        return round(int(match.group(1)) / 100000, 2)

    return None


def extract_location(text):
    text_lower = text.lower()
    for city in TARGET_CITIES:
        if city in text_lower:
            return city.title()
    return None


def extract_college(text):
    text_lower = text.lower()
    for college in TARGET_COLLEGES:
        if college in text_lower:
            return college.upper()
    return None


def extract_contact(text):
    phones = re.findall(r'\b\d{10}\b', text)
    return ", ".join(phones)


# ---------- PROCESS SINGLE FILE ----------
def process_html(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    results = []

    for msg in soup.find_all("div", class_="message"):

        text_div = msg.find("div", class_="text")
        if not text_div:
            continue

        text = format_message(text_div)

        # ❌ GLOBAL EXCLUSION
        if "both are from our yellapragada's groups" in text.lower():
            continue

        # Split multiple profiles inside message
        profiles = re.split(r'(?=👨)', text)

        for p in profiles:
            p = p.strip()

            if not p or "👨" not in p:
                continue

            # ❌ PROFILE LEVEL EXCLUSION
            if "both are from our yellapragada's groups" in p.lower():
                continue

            # ---------- EXTRACT ----------
            name = extract_field(p, "NAME")
            dob = extract_field(p, "DOB")
            salary = extract_salary(p)
            location = extract_location(p)
            college = extract_college(p)
            phone = extract_contact(p)

            year = extract_year(dob)
            age = calculate_age(year)

            # ---------- FILTERS ----------
            if age is None or age > MAX_AGE:
                continue

            if salary is not None and salary < MIN_SALARY:
                continue

            if location is None:
                continue

            if college is None:
                continue

            results.append({
                "Name": name,
                "DOB": dob,
                "Age": age,
                "Salary (LPA)": salary,
                "Location": location,
                "College": college,
                "Phones": phone,
                "Message": p,
                "Source File": file_path
            })

    return results


# ---------- PROCESS ALL FILES ----------
files_list = glob.glob("messages*.html")

print("📂 Files detected:", files_list)

all_results = []

for file in files_list:
    print(f"Processing: {file}")
    file_results = process_html(file)
    all_results.extend(file_results)


# ---------- DEDUPLICATION ----------
seen = set()
unique_results = []

for row in all_results:
    key = (row["Name"], row["Phones"]) if row["Phones"] else (row["Name"],)

    if key not in seen:
        seen.add(key)
        unique_results.append(row)

data = unique_results


# ---------- OUTPUT ----------
print(f"\n✅ Total Unique Matches: {len(data)}")

for i, row in enumerate(data, 1):
    print(f"--- Match {i} ---")
    print(f"Name: {row['Name']}")
    print(f"Age: {row['Age']}")
    print(f"Salary: {row['Salary (LPA)']}")
    print(f"Location: {row['Location']}")
    print(f"College: {row['College']}")
    print(f"Source: {row['Source File']}")
    print("\n--- Message ---")
    print(row["Message"])
    print("\n" + "="*80 + "\n")

df = pd.DataFrame(data)
df
df.to_excel("final_filtered_profiles.xlsx", index=False)

print("✅ Excel created: final_filtered_profiles.xlsx")

📂 Files detected: ['messages2.html', 'messages.html', 'messages3.html']
Processing: messages2.html
Processing: messages.html
Processing: messages3.html

✅ Total Unique Matches: 1
--- Match 1 ---
Name: Komaravolu Adharvan Sameer
Age: 27
Salary: 21.0
Location: Mumbai
College: BITS
Source: messages3.html

--- Message ---
👨🏻‍💼NAME:. Komaravolu Adharvan Sameer
INTIPERU:. Komaravolu
SAKA:. 6000 Niyogi
GOTHRAM:. Bharadwajasa
STAR:. uttarashada
DOB:. 18.1.1999
POB:. Mumbai
TOB:. night 1.55 ....on  17.1.99 ,,,,early hours of 18 /1/99
HEIGHT:. 6 feet
COLOUR:. Fair white
EDU:. B.tech,B.com
JOB:. Indian Army officer
HABITS:. Singing ,Dancing,playing so many
SALARY:. 21 lakhs per annum
PARENTS:. Father: satya prasad Komaravolu, state Bank of India
Mother: Vishnu priyya valluri, State Govt .employee
SIBLINGS:. one Elder Brother working in ITC .....Married
REQUIREMENTS:. No pattimpu
CONTACT:. M: 7837831522,F: 7731931515
✶⋆  🎀  YELLAPRAGADA'S  🎀  ⋆✶


✅ Excel created: final_filtered_profiles.xlsx
